# Corpus Inspection
Sanity-check the chunked corpus and test both retrievers on hand-written queries.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
# Load metadata
meta_path = Path('../corpus/chunks_meta.jsonl')
chunks = []
with open(meta_path) as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print(f'Total chunks: {len(chunks)}')

import pandas as pd
df = pd.DataFrame(chunks)
print(df.groupby('source_doc').size().rename('chunk_count'))
df.head()

In [ ]:
# Token length distribution
import numpy as np
import matplotlib.pyplot as plt

token_lens = [len(c['text'].split()) * 4 // 3 for c in chunks]
plt.figure(figsize=(8, 4))
plt.hist(token_lens, bins=40)
plt.xlabel('Approx tokens')
plt.ylabel('Number of chunks')
plt.title('Chunk length distribution')
plt.axvline(800, color='red', linestyle='--', label='max_chunk_tokens=800')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Mean={np.mean(token_lens):.0f}  p50={np.percentile(token_lens,50):.0f}  p95={np.percentile(token_lens,95):.0f}')

In [ ]:
# Test dense retriever on 10 hand-written queries
from tutor.retriever import DenseRetriever, SparseRetriever

dense = DenseRetriever()
sparse = SparseRetriever()

test_queries = [
    'First-line treatment for uncomplicated malaria in children',
    'Management of severe acute malnutrition',
    'Diagnosis and treatment of pneumonia in adults',
    'Hypertension management in pregnancy',
    'HIV antiretroviral therapy initiation criteria',
    'Postpartum haemorrhage prevention',
    'Appendicitis surgical management',
    'Tuberculosis diagnosis and treatment',
    'Neonatal jaundice management',
    'Type 2 diabetes treatment guidelines',
]

for q in test_queries:
    results = dense.search(q, k=3)
    print(f'\nQ: {q}')
    for r in results:
        print(f'  [{r.score:.3f}] {r.source_doc} — {r.section_title}')

In [ ]:
# Compare dense vs sparse on same queries
for q in test_queries[:3]:
    d = dense.search(q, k=3)
    s = sparse.search(q, k=3)
    print(f'\nQuery: {q}')
    print('  Dense:', [f'{r.source_doc}/{r.section_title}' for r in d])
    print('  Sparse:', [f'{r.source_doc}/{r.section_title}' for r in s])